# 08. Deep Learning & Neural Forecasting Architectures

## 1. Objectives & Theoretical Framework
Deep Neural Networks offer strong non-linear feature interaction capabilities across high-dimensional retail feature spaces:

1. **Continuous Latent Representations**: Deep Multi-Layer Perceptrons (MLPs) project continuous price elasticity, rolling momentum signals, and cyclical calendar embeddings into shared latent space.
2. **Adaptive Gradient Descent Optimization**: Backpropagation with Adam optimizer, batch normalization, and early stopping prevents overfitting on zero-inflated retail series.
3. **Comparative Evaluation**: Benchmarking neural architectures against Gradient Boosted Decision Trees (LightGBM) under identical rolling temporal validation.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add root to sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(root_dir))

from src.data.loader import generate_synthetic_m5_data, load_raw_data
from src.data.preprocess import melt_sales_data, merge_calendar_and_prices
from src.evaluation.backtest import run_backtest
from src.features.pipeline import build_feature_table
from src.models.gbm import LightGBMForecaster
from src.models.neural import MLPDemandForecaster
from src.utils.config import load_config
from src.utils.logger import get_logger

# Styling
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

config = load_config(root_dir / "configs" / "model_config.yaml")
logger = get_logger("neural_notebook")
print(f"Project: {config.project.get('name')} | Neural Architecture Suite")

## 2. Ingestion & Feature Engineering

In [ ]:
SAMPLE_SERIES = 75

try:
    cal_df, prc_df, sal_df = load_raw_data(root_dir / "data" / "raw", nrows=SAMPLE_SERIES)
    print(f"Loaded {len(sal_df)} series from data/raw")
except Exception as e:
    print(f"Raw data not found ({e}). Using synthetic M5 dataset...")
    cal_df, prc_df, sal_df = generate_synthetic_m5_data(
        num_items=75, num_stores=2, num_days=365, random_seed=42
    )

sales_long = melt_sales_data(sal_df)
merged_df = merge_calendar_and_prices(sales_long, cal_df, prc_df)
feat_df = build_feature_table(merged_df)
print(f"Prepared feature dataset: {len(feat_df):,} rows x {len(feat_df.columns)} columns")

## 3. Training Deep MLP Forecaster & Loss Convergence

In [ ]:
split_date = "2016-04-01"
train_df = feat_df[feat_df["date"] < split_date].copy()
val_df = feat_df[feat_df["date"] >= split_date].copy()

mlp_forecaster = MLPDemandForecaster(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=100,
    batch_size=256,
    early_stopping=True,
    random_state=42,
)

mlp_forecaster.fit(train_df)

# Plot Training Loss Curve
plt.figure(figsize=(10, 4))
plt.plot(mlp_forecaster.loss_curve, marker="o", markersize=3, color="#2563eb", linewidth=2)
plt.title(
    "Deep MLP Neural Network: Training Loss Convergence Curve (Adam Optimizer)",
    fontweight="bold",
    fontsize=12,
)
plt.xlabel("Training Epoch / Iteration")
plt.ylabel("Loss (Mean Squared Error)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Head-to-Head Rolling Backtest: MLP vs LightGBM

In [ ]:
lgb_model = LightGBMForecaster(
    n_estimators=150, learning_rate=0.06, num_leaves=31, tweedie_variance_power=1.15
)
mlp_model = MLPDemandForecaster(hidden_layer_sizes=(128, 64, 32), max_iter=80, random_state=42)

print("Evaluating LightGBM Tweedie Model...")
lgb_metrics, lgb_oof, _ = run_backtest(model=lgb_model, df=feat_df, horizon=28, n_splits=3)

print("Evaluating Deep MLP Neural Model...")
mlp_metrics, mlp_oof, _ = run_backtest(model=mlp_model, df=feat_df, horizon=28, n_splits=3)

comparison_df = pd.DataFrame(
    [
        {
            "Model": "LightGBM (Tweedie)",
            "WRMSSE": lgb_metrics["mean_wrmsse"],
            "WAPE": lgb_metrics["mean_wape"],
            "RMSE": lgb_metrics["mean_rmse"],
            "Fit Time (s)": lgb_metrics["total_fit_time_sec"],
        },
        {
            "Model": "Deep MLP (128-64-32)",
            "WRMSSE": mlp_metrics["mean_wrmsse"],
            "WAPE": mlp_metrics["mean_wape"],
            "RMSE": mlp_metrics["mean_rmse"],
            "Fit Time (s)": mlp_metrics["total_fit_time_sec"],
        },
    ]
)
comparison_df

## 5. Forecast Trajectory Overlay

In [ ]:
sample_id = feat_df["id"].iloc[0]
actual_series = feat_df[feat_df["id"] == sample_id].sort_values("date")
lgb_pred_series = lgb_oof[lgb_oof["id"] == sample_id].sort_values("date")
mlp_pred_series = mlp_oof[mlp_oof["id"] == sample_id].sort_values("date")

plt.figure(figsize=(12, 5))
plt.plot(
    actual_series["date"].iloc[-60:],
    actual_series["sales"].iloc[-60:],
    label="Actual Sales",
    color="#1e293b",
    linewidth=2,
)
plt.plot(
    lgb_pred_series["date"],
    lgb_pred_series["y_pred"],
    label="LightGBM Forecast",
    color="#2563eb",
    linestyle="--",
    linewidth=2,
)
plt.plot(
    mlp_pred_series["date"],
    mlp_pred_series["y_pred"],
    label="Deep MLP Forecast",
    color="#10b981",
    linestyle=":",
    linewidth=2,
)

plt.title(
    f"Out-of-Fold 28-Day Demand Trajectory Comparison: {sample_id}", fontweight="bold", fontsize=12
)
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Key Takeaways & Architecture Insights
1. **Tree vs Neural Dynamics**: While Deep MLPs capture complex smooth non-linearities, LightGBM with Tweedie loss demonstrates superior capability in handling zero-inflation and step-change calendar events.
2. **Ensemble Opportunities**: Blending neural MLP representations with LightGBM GBDT predictions provides diversification across erratic intermittent item tiers.